In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install tensorflow==2.16.1

In [ ]:
!pip install ml_dtypes==0.5.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 49.9 MB/s eta 0:00:00
  Attempting uninstall: ml_dtypes
    Found existing installation: ml-dtypes 0.3.2
    Uninstalling ml-dtypes-0.3.2:
      Successfully uninstalled ml-dtypes-0.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.16.1 requires ml-dtypes~=0.3.1, but you have ml-dtypes 0.5.0 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.16.1 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tf-keras 2.20.0 requires tensorflow<2.21,>=2.20, but you have tensorflow 2.16.1 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.16.1 which is incompatible.


In [ ]:
import tensorflow as tf
import numpy as np
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
tf.config.set_visible_devices([], 'GPU')
print("✅ GPU kapatıldı, CPU modunda çalışıyor")

old_model = tf.keras.models.load_model('/content/drive/MyDrive/isaret_dili_beyni_v2.h5')
weights = old_model.get_weights()
print(f"✅ {len(weights)} ağırlık katmanı alındı")
del old_model

with tf.device('/CPU:0'):
    new_model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(60, 211)),
        tf.keras.layers.LSTM(256, return_sequences=True),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.LSTM(128, return_sequences=True),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.LSTM(64, return_sequences=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(226, activation='softmax')
    ])
    new_model.set_weights(weights)

    test = np.random.rand(1, 60, 211).astype(np.float32)
    pred = new_model.predict(test, verbose=0)
    print(f"✅ Test tahmini çalıştı: shape={pred.shape}")

print("\n--- Dönüşüm Başladı ---")
converter = tf.lite.TFLiteConverter.from_keras_model(new_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
converter._experimental_lower_tensor_list_ops = False

tflite_model = converter.convert()

with open('/content/drive/MyDrive/isaret_dili_beyni_v2.tflite', 'wb') as f:
    f.write(tflite_model)

print(f"✅ Tamamlandı! Boyut: {len(tflite_model) / 1024 / 1024:.2f} MB")

✅ GPU kapatıldı, CPU modunda çalışıyor


✅ 25 ağırlık katmanı alındı
✅ Test tahmini çalıştı: shape=(1, 226)

--- Dönüşüm Başladı ---


TypeError: 'NoneType' object is not callable

In [ ]:
import tensorflow as tf
import numpy as np
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
tf.config.set_visible_devices([], 'GPU')
print("✅ GPU kapatıldı")

old_model = tf.keras.models.load_model('/content/drive/MyDrive/isaret_dili_beyni_v2.h5')
weights = old_model.get_weights()
print(f"✅ {len(weights)} ağırlık katmanı alındı")
del old_model

with tf.device('/CPU:0'):
    new_model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(60, 211)),
        tf.keras.layers.LSTM(256, return_sequences=True),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.LSTM(128, return_sequences=True),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.LSTM(64, return_sequences=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(226, activation='softmax')
    ])
    new_model.set_weights(weights)
    print("✅ Ağırlıklar aktarıldı")

# Önce SavedModel olarak kaydet
saved_model_path = '/content/isaret_saved_model'
new_model.export(saved_model_path)
print("✅ SavedModel kaydedildi")

# SavedModel'den TFLite'a dönüştür
converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_path)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
converter._experimental_lower_tensor_list_ops = False

tflite_model = converter.convert()

with open('/content/drive/MyDrive/isaret_dili_beyni_v2.tflite', 'wb') as f:
    f.write(tflite_model)

print(f"✅ Tamamlandı! Boyut: {len(tflite_model) / 1024 / 1024:.2f} MB")

✅ GPU kapatıldı


✅ 25 ağırlık katmanı alındı
✅ Ağırlıklar aktarıldı
Saved artifact at '/content/isaret_saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 60, 211), dtype=tf.float32, name='keras_tensor_169')
Output Type:
  TensorSpec(shape=(None, 226), dtype=tf.float32, name=None)
Captures:
  134904014748368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134904011933648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134904008117712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134904008116368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134904008118672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134904008118096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134904008117520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134904008117328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134904008115600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134904008114832: Ten

In [2]:
import tensorflow as tf
import numpy as np
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
tf.config.set_visible_devices([], 'GPU')

old_model = tf.keras.models.load_model('/content/drive/MyDrive/isaret_dili_beyni_v2.h5')
weights = old_model.get_weights()
del old_model

new_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(60, 211)),
    tf.keras.layers.LSTM(256, return_sequences=True),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.LSTM(128, return_sequences=True),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.LSTM(64, return_sequences=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(226, activation='softmax')
])
new_model.set_weights(weights)

# Dönüştür
converter = tf.lite.TFLiteConverter.from_keras_model(new_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
converter._experimental_lower_tensor_list_ops = False

# ÖNEMLİ: uyumlu ops için
converter.target_spec.supported_types = [tf.float32]

tflite_model = converter.convert()

with open('/content/drive/MyDrive/isaret_dili_beyni_v2_android.tflite', 'wb') as f:
    f.write(tflite_model)

print(f"✅ Boyut: {len(tflite_model)/1024/1024:.2f} MB")

Saved artifact at '/tmp/tmpbt18qxkw'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 60, 211), dtype=tf.float32, name='keras_tensor_78')
Output Type:
  TensorSpec(shape=(None, 226), dtype=tf.float32, name=None)
Captures:
  138322893723344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138322893719888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138322893725264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138322893725072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138322893724112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138322893724304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138322893724688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138322893725456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138322893725840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138322893726608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1383228937264